In [1]:
import numpy as np 
import pandas as pd

In [8]:
from sklearn.model_selection import train_test_split


from sklearn.datasets import load_breast_cancer
from sklearn.tree import DecisionTreeClassifier

from collections import Counter

In [9]:
x, y = load_breast_cancer(return_X_y=True)

In [10]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [ ]:
class Node:

    def __init__(self):
        self.feature = None
        self.threshold = None
        self.left = None  
        self.right = None 
        self.value = None 
        self.gini = 0.0   
        self.n_samples = 0

    def is_leaf(self):
        return self.value is not None

class decision_tree:
    
    def __init__(self, max_depth=None, min_samples_split=2, min_samples_leaf=1):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.root = None

    def _gini(self, y):
        
        n = len(y)
        if n == 0: return 0.0
        counts = Counter(y)
        return 1.0 - sum((c/n)**2 for c in counts.values())

    def _best_split(self, X, y):
        
        n, p = X.shape
        best_gain = -1
        best_feat, best_thresh = None, None
        parent_gini = self._gini(y)

        for j in range(p):
            thresholds = np.unique(X[:, j])
            for t in thresholds:
                left_mask = X[:, j] <= t
                right_mask = ~left_mask
                nL, nR = left_mask.sum(), right_mask.sum()

                if nL < self.min_samples_leaf or nR < self.min_samples_leaf:
                    continue

                gL = self._gini(y[left_mask])
                gR = self._gini(y[right_mask])
                weighted = (nL/n)*gL + (nR/n)*gR

                gain = parent_gini - weighted
                if gain > best_gain:
                    best_gain = gain
                    best_feat = j
                    best_thresh = t

        return best_feat, best_thresh, best_gain

    def _build(self, X, y, depth):

        node = Node()
        node.gini = self._gini(y)
        node.n_samples = len(y)

        should_stop = (
            node.gini == 0 
            or len(y) < self.min_samples_split
            or (self.max_depth is not None
                and depth >= self.max_depth)  
        )
        if should_stop:
            node.value = Counter(y).most_common(1)[0][0]
            return node

        feat, thresh, gain = self._best_split(X, y)
        if gain <= 0:
            node.value = Counter(y).most_common(1)[0][0]
            return node

        node.feature = feat
        node.threshold = thresh
        mask = X[:, feat] <= thresh
        node.left  = self._build(X[mask],  y[mask],  depth+1)
        node.right = self._build(X[~mask], y[~mask], depth+1)
        return node

    def fit(self, X, y):
        self.root = self._build(np.asarray(X), np.asarray(y), depth=0)
        return self

    def _predict_one(self, x, node):

        if node.is_leaf():
            return node.value
        if x[node.feature] <= node.threshold:
            return self._predict_one(x, node.left)
        return self._predict_one(x, node.right)

    def predict(self, X):
        return np.array([self._predict_one(x, self.root) for x in X])

    def score(self, X, y):
        return np.mean(self.predict(X) == y)

In [13]:
dt = decision_tree()
dt.fit(x_train, y_train)

In [15]:
dt.score(x_test, y_test)

np.float64(0.9298245614035088)

In [5]:
dt = DecisionTreeClassifier()
dt.fit(x_train, y_train)

DecisionTreeClassifier()

In [7]:
dt.score(x_test, y_test)

0.9385964912280702